In [58]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.optim import Adam
from tqdm import tqdm

# Глянем на датасет

In [59]:


# загрузка датасета
raw = pd.read_csv('../data/yelp_reviews.csv')

texts = raw['text']
labels = raw['label']


print('\nРазмер датасета:')
print(len(texts))


print('\nСтатистика значений меток-оценок:')
print(pd.Series(labels).value_counts())


print('\n', '='*50, '\n')

print('Примеры текстов и их меток-оценок:')
print('-'*35)
for i in range(0, 5):
    print(labels[i], '  ', '  '.join(texts[i].split('\\n')))


Размер датасета:
6500

Статистика значений меток-оценок:
label
2    1483
1    1429
3    1327
0    1132
4    1129
Name: count, dtype: int64


Примеры текстов и их меток-оценок:
-----------------------------------
0    Worst sandwich on Earth.  I'd rather eat a dead whore.  Please...never come here.
0    First time in Pittsburgh, from Chicago.     1. NO ONE SEEMS TO KNOW HOW THE PUBLIC TRANSPORTATION SYSTEM WORKS.  2. What is with the silly system of paying on your way out sometimes? It seems like the dumbest idea ever.  3. Where are the reloadable transit cards?  4. Why are the conductors so incredibly angry?  5. Why do I need to request a stop on the \"T\"?    Of the public transportation systems I've used, this is definitely the most confusing so far. Even if the website had an easier to understand fare explanation.    Learn from the CTA. I have a new appreciation for Chicago Transit Authority.
0    Worst experience at a restaurant ever.  I don't need anything fancy but this was path

# Задание 1
Что нужно сделать
* Подготовьте датасет. 
* Чтобы сформировать батчи, используйте кастомную функцию collate_fn. 
* Подготовка датасета проводится аналогично подготовке датасета для токенизации текста. 
* При инициализации датасета нужно сохранить аргументы из конструктора, делать дополнительные преобразования не нужно.
* В методе `__getitem__` нужно возвращать объекты класса `torch.tensor`. 
* Текст можно обрезать обычной срезкой списка в питоне `[:self.max_len]`.
* В кастомной функции `collate_fn` для пэддинга используйте метод `pad_sequence`.

## разделение выборки на трейн и тест

In [60]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
                                                   texts, labels, test_size=0.2, random_state=42)

train_texts.shape, val_texts.shape, train_labels.shape, val_labels.shape

((5200,), (1300,), (5200,), (1300,))

## создание токенизатора с помощью класса AutoTokenizer

In [61]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

### токенизируем тексты

In [62]:
train_texts_tokenized = tokenizer(train_texts.tolist(), truncation=True)['input_ids']
val_texts_tokenized = tokenizer(val_texts.tolist(), truncation=True)['input_ids']

len(val_texts_tokenized), len(train_texts_tokenized)

(1300, 5200)

### смотрим как преобразовывался исходный текст

In [63]:
print('Текст         :', train_texts.tolist()[0][:36])
print('Сами токены   :', tokenizer.tokenize(train_texts.tolist()[0])[:10])
print('Номера токенов:', val_texts_tokenized[0][0:10])

Текст         : I wouldn't pay attention to the Trib
Сами токены   : ['i', 'wouldn', "'", 't', 'pay', 'attention', 'to', 'the', 'tri', '##b']
Номера токенов: [101, 2023, 2173, 3513, 999, 2155, 3079, 1998, 2448, 1010]


### что еще возвращает токенизатор

In [64]:
full_res = tokenizer(val_texts.tolist(), truncation=True)
type(full_res), full_res.keys()

(transformers.tokenization_utils_base.BatchEncoding,
 dict_keys(['input_ids', 'token_type_ids', 'attention_mask']))

In [65]:
full_res['input_ids']

[[101,
  2023,
  2173,
  3513,
  999,
  2155,
  3079,
  1998,
  2448,
  1010,
  2009,
  1005,
  1055,
  1037,
  2307,
  5101,
  2173,
  2000,
  4521,
  1012,
  1996,
  15890,
  2015,
  1010,
  2192,
  2081,
  1998,
  2081,
  2000,
  2344,
  2024,
  3565,
  26255,
  2135,
  2204,
  1012,
  1045,
  2293,
  2008,
  2027,
  2123,
  1005,
  1056,
  2031,
  22201,
  999,
  2123,
  1005,
  1056,
  2131,
  2033,
  3308,
  1010,
  1045,
  2293,
  22201,
  2021,
  2040,
  3791,
  14744,
  3514,
  1010,
  2214,
  3514,
  1010,
  2062,
  4933,
  1029,
  1996,
  15890,
  2003,
  2204,
  2438,
  1012,
  2027,
  1005,
  2128,
  2124,
  2205,
  2005,
  2307,
  3869,
  1998,
  21475,
  2015,
  2295,
  1045,
  2018,
  15890,
  2044,
  15890,
  2296,
  2051,
  1045,
  2253,
  1012,
  1032,
  9152,
  2253,
  2007,
  2814,
  1010,
  2178,
  2051,
  2007,
  2026,
  4268,
  1010,
  5681,
  2011,
  2870,
  1998,
  2938,
  2012,
  1996,
  3347,
  1012,
  2467,
  2204,
  1012,
  1996,
  3524,
  2064,
  2022,
  

In [66]:
full_res['token_type_ids']

[[0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,

In [67]:
full_res['attention_mask']

[[1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,

## создаём класс кастомного датасета, наследуясь от класса Dataset из PyTorch

In [68]:
class YelpDataset(Dataset):
    # в конструкторе просто сохраняем тексты и классы
    def __init__(self, texts, labels, max_len=256):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len


    # возвращаем размер датасета (кол-во текстов)
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        # возвращаем текст и его класс
        # для текста ограничиваем длину (NB!)
        # не делаем никаких доп. преобразований как padding и masking
        return {
            'text': torch.tensor(self.texts[idx][:self.max_len], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

## кастомная функция collate_fn для формирования батчей

In [69]:
def collate_fn(batch):
    texts = [torch.tensor(item['text']) for item in batch]
    labels = torch.tensor([item['label'] for item in batch])
    lengths = torch.tensor([len(seq) for seq in texts])
    padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)


    return {
        'input_ids': padded_texts, 
        'lengths': lengths, 
        'labels': labels
    }

In [70]:
train_dataset = YelpDataset(texts=train_texts_tokenized, labels=train_labels.tolist())
val_dataset = YelpDataset(texts=val_texts_tokenized, labels=val_labels.tolist())

val_dataset[0]

{'text': tensor([  101,  2023,  2173,  3513,   999,  2155,  3079,  1998,  2448,  1010,
          2009,  1005,  1055,  1037,  2307,  5101,  2173,  2000,  4521,  1012,
          1996, 15890,  2015,  1010,  2192,  2081,  1998,  2081,  2000,  2344,
          2024,  3565, 26255,  2135,  2204,  1012,  1045,  2293,  2008,  2027,
          2123,  1005,  1056,  2031, 22201,   999,  2123,  1005,  1056,  2131,
          2033,  3308,  1010,  1045,  2293, 22201,  2021,  2040,  3791, 14744,
          3514,  1010,  2214,  3514,  1010,  2062,  4933,  1029,  1996, 15890,
          2003,  2204,  2438,  1012,  2027,  1005,  2128,  2124,  2205,  2005,
          2307,  3869,  1998, 21475,  2015,  2295,  1045,  2018, 15890,  2044,
         15890,  2296,  2051,  1045,  2253,  1012,  1032,  9152,  2253,  2007,
          2814,  1010,  2178,  2051,  2007,  2026,  4268,  1010,  5681,  2011,
          2870,  1998,  2938,  2012,  1996,  3347,  1012,  2467,  2204,  1012,
          1996,  3524,  2064,  2022,  4689, 

In [71]:
batch_size = 64

In [72]:

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [73]:
print(f'Количество батчей в train_dataloader: {len(train_dataloader)}')
print(f'Количество батчей в val_dataloader: {len(val_dataloader)}')


print('Размерности батчей:')
for batch in train_dataloader:
    print('input_ids:', batch['input_ids'].shape)
    print('lengths:', batch['lengths'].shape)
    print('labels:', batch['labels'].shape)
    break

Количество батчей в train_dataloader: 82
Количество батчей в val_dataloader: 21
Размерности батчей:
input_ids: torch.Size([64, 256])
lengths: torch.Size([64])
labels: torch.Size([64])


C:\Users\aseva\AppData\Local\Temp\ipykernel_2188\2984711182.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(item['text']) for item in batch]


# Задание 2
Что нужно сделать
* Напишите код обучения модели. 
* Используйте класс RNN в forward-проходе. 
* Не забудьте, что внутри метода forward нужно упаковать батч текстов в объект `PackedSequence` и подать его на вход рекуррентному блоку.
* В качестве RNN-слоя модели следует использовать `nn.RNN`. Он должен принимать на вход вектор размера `embedding_dim` и считать скрытое состояние размера `hidden_size`.
* В качестве финального линейного fc (fully-connected) слоя модели следует использовать обычный линейный слой nn.Linear, принимающий на вход вектор размера `hidden_size` и выдающий в качестве выхода вектор размера `output_size`.
Чтобы посчитать скоры классификации, нужно применить линейный fully-connected слой к последнему скрытому состоянию, то есть к hidden[-1].
* Для получения разных элементов батча следует вспомнить, что батч в нашем случае — это словарь. Тогда входные токены можно достать как batch['input_ids], длины как batch['lengths'] и т. д.
* Обновление весов и подсчёт функции потерь при обучении RNN при написании кода ничем не отличается от других нейросетей.

## class SimpleRNN definition

In [74]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)


    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)
        packed = torch.nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_output, hidden = self.rnn(packed)
        output, _ = torch.nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)
        
        # Используем последнее скрытое состояние для классификации
        out = self.fc(hidden[-1])
        return out

## создаем модель, привязываем оптимайзер и выбираем функцию потерь

In [75]:
vocab_size = tokenizer.vocab_size
model = SimpleRNN(vocab_size, embedding_dim=128, hidden_size=256, output_size=5)
loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

## training loop

In [76]:
train_losses, val_losses, val_accu = [], [], []
n_epochs = 10


for epoch in range(n_epochs):

    model.train()
    total_train_loss = 0.
    for batch in train_dataloader:
        inputs = batch['input_ids']
        lengths = batch['lengths']
        labels = batch['labels']


        optimizer.zero_grad()
        outputs = model(inputs, lengths)
        loss = loss_fn(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()


        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_dataloader)
    train_losses.append(avg_train_loss)
    

    model.eval()
    total_val_loss = 0.
    correct_val = 0
    total_val = 0
    for batch in val_dataloader:
        inputs = batch['input_ids']
        lengths = batch['lengths']
        labels = batch['labels']


        with torch.no_grad():
            outputs = model(inputs, lengths)
            loss = loss_fn(outputs, labels)
            total_val_loss += loss.item()
            correct_val += (np.argmax(outputs.numpy(), axis=1) == labels.numpy()).sum()
            total_val += len(outputs)

    avg_val_loss = total_val_loss / len(val_dataloader)
    accuracy_val = correct_val / total_val
    val_losses.append(avg_val_loss)
    val_accu.append(accuracy_val)

    print(f"Epoch {epoch + 1}, Train Loss: {avg_train_loss:.4f}, Val loss: {avg_val_loss:.4f},",
          f"Val Accuracy: {accuracy_val*100} %")

C:\Users\aseva\AppData\Local\Temp\ipykernel_2188\2984711182.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(item['text']) for item in batch]


Epoch 1, Train Loss: 1.6048, Val loss: 1.5935, Val Accuracy: 24.76923076923077 %
Epoch 2, Train Loss: 1.5229, Val loss: 1.6266, Val Accuracy: 26.23076923076923 %
Epoch 3, Train Loss: 1.4310, Val loss: 1.6180, Val Accuracy: 31.153846153846153 %
Epoch 4, Train Loss: 1.3230, Val loss: 1.6809, Val Accuracy: 28.692307692307693 %
Epoch 5, Train Loss: 1.2056, Val loss: 1.7197, Val Accuracy: 29.53846153846154 %
Epoch 6, Train Loss: 1.0769, Val loss: 1.8493, Val Accuracy: 29.230769230769234 %
Epoch 7, Train Loss: 0.9343, Val loss: 2.0033, Val Accuracy: 29.69230769230769 %
Epoch 8, Train Loss: 0.7679, Val loss: 2.1939, Val Accuracy: 28.692307692307693 %
Epoch 9, Train Loss: 0.6246, Val loss: 2.4143, Val Accuracy: 29.153846153846153 %
Epoch 10, Train Loss: 0.4763, Val loss: 2.6182, Val Accuracy: 29.615384615384617 %
